In [11]:
# to import mlflow and know the version and trackinig 
import mlflow 
from mlflow import MlflowClient
print("MLflow version:", mlflow.__version__)
print("MLflow tracking URI:",mlflow.get_tracking_uri())

MLflow version: 3.10.1
MLflow tracking URI: sqlite:///c:\Users\PC\codegpt\MLOps\mlruns.db


In [12]:
# to store the model in mlflow and use SQLite Backend store and file store for artifacts

from pathlib import Path

PROJECT_DIR =Path.cwd()
DB_PATH = PROJECT_DIR / "mlruns.db"


TRACKING_URI = f"sqlite:///{DB_PATH}"
mlflow.set_tracking_uri(TRACKING_URI)


print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("MLflow artifact URI:", mlflow.get_artifact_uri())


 

MLflow tracking URI: sqlite:///c:\Users\PC\codegpt\MLOps\mlruns.db
MLflow artifact URI: file:///c:/Users/PC/codegpt/MLOps/mlruns/0/caf0f4b336d841dcb37f6f7b303ab605/artifacts


In [15]:
# to bild the Experiment and set the experiment name and get the experiment id
EXPERIMENT_NAME = "my_experiment"

# set_experiment = create_experiment if not exist
mlflow.set_experiment(EXPERIMENT_NAME)

# get_experiment_by_name
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

print("Experiment Name:", experiment.name)
print("Experiment ID:", experiment.experiment_id)


Experiment Name: my_experiment
Experiment ID: 1


In [27]:

# create a smallest useful Mlflow run
with mlflow.start_run(run_name="my_run_1") as run:
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_metric("accuracy", 0.95)
    mlflow.set_tag("tag1", "value1")
  
    print("Run ID:", run.info.run_id)
    print("Run Info:", run.info)

    run_id = run.info.run_id
    experiment_id = run.info.experiment_id
    
    
print("Run ID:", run_id)
print("Experiment ID:", experiment_id)
print("Run URI:", mlflow.get_run(run_id))
mlflow.end_run()

Run ID: 1d78dc0a672a4b90bc7f3d75d1c31670
Run Info: <RunInfo: artifact_uri='file:///c:/Users/PC/codegpt/MLOps/mlruns/1/1d78dc0a672a4b90bc7f3d75d1c31670/artifacts', end_time=None, experiment_id='1', lifecycle_stage='active', run_id='1d78dc0a672a4b90bc7f3d75d1c31670', run_name='my_run_1', start_time=1788188785899, status='RUNNING', user_id='PC'>
Run ID: 1d78dc0a672a4b90bc7f3d75d1c31670
Experiment ID: 1
Run URI: <Run: data=<RunData: metrics={'accuracy': 0.95}, params={'learning_rate': '0.1'}, tags={'mlflow.runName': 'my_run_1',
 'mlflow.source.name': 'mlflow.ipynb',
 'mlflow.source.type': 'NOTEBOOK',
 'mlflow.user': 'PC',
 'tag1': 'value1'}>, info=<RunInfo: artifact_uri='file:///c:/Users/PC/codegpt/MLOps/mlruns/1/1d78dc0a672a4b90bc7f3d75d1c31670/artifacts', end_time=1788188785961, experiment_id='1', lifecycle_stage='active', run_id='1d78dc0a672a4b90bc7f3d75d1c31670', run_name='my_run_1', start_time=1788188785899, status='FINISHED', user_id='PC'>, inputs=<RunInputs: dataset_inputs=[], model

In [28]:
# log single and multiple parametrs and metrics
with mlflow.start_run(run_name="my_run_2") as run:
    mlflow.log_param("Algorithm","Random_Forest")
    mlflow.log_params({
        "learing_rate": 0.01,
        "batch_size": 32,
        "max_depth": 5,
        "random_state": 42
    })
    
mlflow.end_run()
    

In [30]:
# log files and structured  artifacts


artifact_dir = Path("mlflow_demo_artifacts")
artifact_dir.mkdir(exist_ok=True)


text_file = artifact_dir / "notes.txt"
text_file.write_text("This file was produced during an MLflow run.\n", encoding="utf-8")

report = {
    "purpose": "MLflow artifact demo",
    "status": "complete",
    "important": True,
}



with mlflow.start_run(run_name="artifacts_demo"):
    mlflow.log_artifact(str(text_file), artifact_path="files")
    mlflow.log_dict(report, "reports/report.json")

    print("Artifacts logged.")
    
mlflow.end_run()


Artifacts logged.


In [32]:
# save a image from matplotlib and log it as an artifact
import matplotlib.pyplot as plt 

with mlflow.start_run(run_name="figure_demo") as run:
    steps = list(range(10))
    loss = [1 / (i + 1) for i in steps]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(steps, loss)
    ax.set_title("Training Loss")
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.grid(True)

    mlflow.log_figure(fig, "plots/training_loss.png")
    plt.close(fig)
    print("Figure logged as an artifact.")

Figure logged as an artifact.


In [33]:
# Cell 9 — compare multiple real ML models

import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

data = load_diabetes()
X_train, X_test, y_train, y_test = train_test_split(
    data.data,
    data.target,
    test_size=0.2,
    random_state=42,
)

models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=50,random_state=42,),
}

for model_name, model in models.items():
    with mlflow.start_run(run_name=model_name):
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

        mse = mean_squared_error(y_test, predictions)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_test, predictions)
        r2 = r2_score(y_test, predictions)

        mlflow.log_params(model.get_params())
        mlflow.log_metrics({
            "mse": mse,
            "rmse": rmse,
            "mae": mae,
            "r2": r2,
        })

        mlflow.set_tag("dataset", "sklearn_diabetes")
        mlflow.set_tag("model_name", model_name)

        print(f"{model_name:18s} RMSE={rmse:.3f}, R2={r2:.3f}")

LinearRegression   RMSE=53.853, R2=0.453
RandomForest       RMSE=55.174, R2=0.425


In [ ]:
from mlflow.models import infer_signature

model = RandomForestRegressor(n_estimators=50, random_state=42)

with mlflow.start_run(run_name="RandomForestRegressor") as run :
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
     
    signature = infer_signature(X_train,predictions)
    
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="random_forest_model",
        signature=signature,
    )
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    mlflow.log_params(model.get_params())
    mlflow.log_metrics({
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
    })

    mlflow.set_tag("dataset", "sklearn_diabetes")
    mlflow.set_tag("model_name", "RandomForestRegressor")
    
    print("\nRun ID:", run.info.run_id)
    print("\nModel URI:", model_info.model_uri)
    print(f"\nRandomForestRegressor RMSE={rmse:.3f}, R2={r2:.3f}")

2026/08/31 17:34:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/31 17:34:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Run ID: 3ee177a9ef35485bb1687afa7e64c6f5

Model URI: models:/m-10f862f4ca89480eb312454c251eac91

\RandomForestRegressor RMSE=55.174, R2=0.425
